# Data Cleaning - Capstone Project

**Author:** Milo Joseph Gaida Barlafante

This notebook loads ten raw data sources, cleans and standardises each one, and merges them into a single daily master dataset covering oil prices, commodity markets, financial indicators, and global conflict events from 1986 to 2025.

### Data Sources

1. **WTI Crude Oil** — Daily spot prices from the EIA via FRED (CSV, 1986–2025)
2. **Henry Hub Natural Gas** — Daily spot prices via FRED (CSV, 1997–2026)
3. **Brent Crude Oil** — Daily futures prices via Yahoo Finance (`BZ=F`)
4. **Gold Futures** — Daily gold prices via Yahoo Finance (`GC=F`)
5. **Copper Futures** — Daily copper prices via Yahoo Finance (`HG=F`)
6. **VIX Volatility Index** — CBOE daily volatility index via Yahoo Finance (`^VIX`)
7. **US Dollar Index (DXY)** — Daily dollar index via Yahoo Finance (`DX-Y.NYB`)
8. **Geopolitical Risk (GPR) Index** — Monthly newspaper-based geopolitical risk scores (Excel, 1900–2026)
9. **ACLED Political Violence Events** — Monthly political violence event counts by country (Excel, 1997–2026)
10. **ACLED Reported Fatalities** — Annual reported conflict fatalities by country (Excel, 1997–2026)

## 1. Imports

All the Python libraries used in this notebook are loaded here before anything else. `pandas` and `numpy` handle data manipulation; `yfinance` pulls financial market prices directly from Yahoo Finance; `sklearn` and `statsmodels` are imported for modelling work later; and `warnings` is silenced to keep the output tidy.

In [28]:
#Data Sets
import yfinance as yf

# Core data handling
import pandas as pd
import numpy as np
import os

# Visualisation
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

# Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.cluster import KMeans
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

# Time series
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# Feature engineering
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Display settings
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', lambda x: '%.3f' % x)

print("Libraries loaded successfully")

Libraries loaded successfully


## 2. Loading Market Data via Yahoo Finance

The five market data series that are not available as flat files are downloaded directly using the `yfinance` library. These cover Brent crude oil, gold, copper, the VIX volatility index, and the US Dollar Index (DXY). The date range is set to 1986–2025, though some series start later — Brent futures data only goes back to around 1990.

In [29]:
start_date = '1986-01-01'
end_date = '2025-12-31'

# Crude oil prices
brent_raw = yf.download('BZ=F', start=start_date, end=end_date, auto_adjust=True)
print("Brent rows:", len(brent_raw))

# Gold prices
gold_raw = yf.download('GC=F', start=start_date, end=end_date, auto_adjust=True)
print("Gold rows:", len(gold_raw))

# Copper prices
copper_raw = yf.download('HG=F', start=start_date, end=end_date, auto_adjust=True)
print("Copper rows:", len(copper_raw))

# VIX (market fear index)
vix_raw = yf.download('^VIX', start=start_date, end=end_date, auto_adjust=True)
print("VIX rows:", len(vix_raw))

# US Dollar Index
dxy_raw = yf.download('DX-Y.NYB', start=start_date, end=end_date, auto_adjust=True)
print("DXY rows:", len(dxy_raw))

[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed
[*********************100%***********************]  1 of 1 completed


Brent rows: 4584
Gold rows: 6357
Copper rows: 6362
VIX rows: 9066


[*********************100%***********************]  1 of 1 completed

DXY rows: 10218


## 3. Loading CSV and Excel Sources

The remaining data sources come as downloaded files. WTI crude oil and Henry Hub natural gas prices are CSV exports from FRED (the St. Louis Fed data repository). The Geopolitical Risk (GPR) index and all four ACLED conflict datasets come as Excel files. Each file is read in and briefly inspected to confirm the shape and column names look correct.

In [30]:
wti_raw = pd.read_csv('WTI_1986_2025.csv')

print(wti_raw.head())
print("WTI shape:", wti_raw.shape)
print("Column names:", wti_raw.columns.tolist())

  observation_date  DCOILWTICO
0       1986-01-02      25.560
1       1986-01-03      26.000
2       1986-01-06      26.530
3       1986-01-07      25.850
4       1986-01-08      25.870
WTI shape: (10436, 2)
Column names: ['observation_date', 'DCOILWTICO']


In [31]:
natgas_raw = pd.read_csv('Henry_Hub_1997_2026.csv')

print(natgas_raw.head())
print("Natural gas shape:", natgas_raw.shape)
print("Column names:", natgas_raw.columns.tolist())

  observation_date  DHHNGSP
0       1997-01-07    3.820
1       1997-01-08    3.800
2       1997-01-09    3.610
3       1997-01-10    3.920
4       1997-01-13    4.000
Natural gas shape: (7645, 2)
Column names: ['observation_date', 'DHHNGSP']


In [32]:
!pip install xlrd
gpr_raw = pd.read_excel('data_gpr_export.xls')

print(gpr_raw.head())
print("GPR shape:", gpr_raw.shape)
print("Column names:", gpr_raw.columns.tolist())

       month  GPR  GPRT  GPRA   GPRH  GPRHT   GPRHA  SHARE_GPR  N10  \
0 1900-01-01  NaN   NaN   NaN 87.928 64.717 110.454        NaN  NaN   
1 1900-02-01  NaN   NaN   NaN 86.566 71.937  96.250        NaN  NaN   
2 1900-03-01  NaN   NaN   NaN 72.141 57.476  84.499        NaN  NaN   
3 1900-04-01  NaN   NaN   NaN 54.419 37.327  65.858        NaN  NaN   
4 1900-05-01  NaN   NaN   NaN 64.405 48.200  74.374        NaN  NaN   

   SHARE_GPRH   N3H  GPRH_NOEW  GPR_NOEW  GPRH_AND  GPR_AND  GPRH_BASIC  \
0       3.172  7724     92.190       NaN   116.949      NaN      84.108   
1       3.123  7173     91.032       NaN   131.075      NaN      79.700   
2       2.602  7762     77.068       NaN   112.939      NaN      65.701   
3       1.963  7488     61.985       NaN    99.778      NaN      53.790   
4       2.323  7360     64.238       NaN   105.352      NaN      55.609   

   GPR_BASIC  SHAREH_CAT_1  SHAREH_CAT_2  SHAREH_CAT_3  SHAREH_CAT_4  \
0        NaN         0.518         0.013         0

In [33]:
acled_violence_raw = pd.read_excel('number_of_political_violence_events_by_country-month-year_as-of-01May2026.xlsx')

print(acled_violence_raw.head())
print("ACLED violence shape:", acled_violence_raw.shape)
print("Column names:", acled_violence_raw.columns.tolist())

       COUNTRY     MONTH  YEAR  EVENTS
0  Afghanistan   January  2017     865
1  Afghanistan  February  2017     697
2  Afghanistan     March  2017    1166
3  Afghanistan     April  2017    1079
4  Afghanistan       May  2017    1229
ACLED violence shape: (28781, 4)
Column names: ['COUNTRY', 'MONTH', 'YEAR', 'EVENTS']


In [34]:
acled_fatalities_raw = pd.read_excel('number_of_reported_fatalities_by_country-year_as-of-01May2026.xlsx')

print(acled_fatalities_raw.head())
print("ACLED fatalities shape:", acled_fatalities_raw.shape)
print("Column names:", acled_fatalities_raw.columns.tolist())

       COUNTRY  YEAR  FATALITIES
0  Afghanistan  2017       36360
1  Afghanistan  2018       42991
2  Afghanistan  2019       41419
3  Afghanistan  2020       31037
4  Afghanistan  2021       42425
ACLED fatalities shape: (2947, 3)
Column names: ['COUNTRY', 'YEAR', 'FATALITIES']


In [35]:
acled_mena_raw = pd.read_excel('Middle-East_aggregated_data_up_to_week_of-2026-04-25.xlsx')

print(acled_mena_raw.head())
print("ACLED MENA shape:", acled_mena_raw.shape)
print("Column names:", acled_mena_raw.columns.tolist())

        WEEK       REGION  COUNTRY   ADMIN1                  EVENT_TYPE  \
0 2016-02-06  Middle East  Bahrain  Capital                     Battles   
1 2026-02-28  Middle East  Bahrain  Capital  Explosions/Remote violence   
2 2026-03-07  Middle East  Bahrain  Capital  Explosions/Remote violence   
3 2026-03-14  Middle East  Bahrain  Capital  Explosions/Remote violence   
4 2026-03-21  Middle East  Bahrain  Capital  Explosions/Remote violence   

     SUB_EVENT_TYPE  EVENTS  FATALITIES  POPULATION_EXPOSURE  \
0       Armed clash       1           0            69821.000   
1  Air/drone strike       9           0           144910.000   
2  Air/drone strike       7           1            57503.000   
3  Air/drone strike       1           0            22606.000   
4  Air/drone strike       1           0            22606.000   

        DISORDER_TYPE      ID  CENTROID_LATITUDE  CENTROID_LONGITUDE  
0  Political violence 285.000             26.193              50.551  
1  Political violence 

In [36]:
acled_europe_raw = pd.read_excel('Europe-Central-Asia_aggregated_data_up_to_week_of-2026-04-25.xlsx')

print(acled_europe_raw.head())
print("ACLED Europe/Central Asia shape:", acled_europe_raw.shape)
print("Column names:", acled_europe_raw.columns.tolist())

        WEEK       REGION  COUNTRY   ADMIN1                  EVENT_TYPE  \
0 2016-02-06  Middle East  Bahrain  Capital                     Battles   
1 2026-02-28  Middle East  Bahrain  Capital  Explosions/Remote violence   
2 2026-03-07  Middle East  Bahrain  Capital  Explosions/Remote violence   
3 2026-03-14  Middle East  Bahrain  Capital  Explosions/Remote violence   
4 2026-03-21  Middle East  Bahrain  Capital  Explosions/Remote violence   

     SUB_EVENT_TYPE  EVENTS  FATALITIES  POPULATION_EXPOSURE  \
0       Armed clash       1           0            69821.000   
1  Air/drone strike       9           0           144910.000   
2  Air/drone strike       7           1            57503.000   
3  Air/drone strike       1           0            22606.000   
4  Air/drone strike       1           0            22606.000   

        DISORDER_TYPE      ID  CENTROID_LATITUDE  CENTROID_LONGITUDE  
0  Political violence 285.000             26.193              50.551  
1  Political violence 

In [37]:
print("=== ALL DATA SOURCES LOADED ===")
print(f"WTI oil:              {wti_raw.shape[0]:>7} rows")
print(f"Natural gas:          {natgas_raw.shape[0]:>7} rows")
print(f"Brent crude:          {brent_raw.shape[0]:>7} rows")
print(f"Gold:                 {gold_raw.shape[0]:>7} rows")
print(f"Copper:               {copper_raw.shape[0]:>7} rows")
print(f"VIX:                  {vix_raw.shape[0]:>7} rows")
print(f"DXY dollar index:     {dxy_raw.shape[0]:>7} rows")
print(f"GPR index:            {gpr_raw.shape[0]:>7} rows")
print(f"ACLED violence:       {acled_violence_raw.shape[0]:>7} rows")
print(f"ACLED fatalities:     {acled_fatalities_raw.shape[0]:>7} rows")
print(f"ACLED MENA:           {acled_mena_raw.shape[0]:>7} rows")
print(f"ACLED Europe/C.Asia:  {acled_europe_raw.shape[0]:>7} rows")
print("\nReady to start cleaning!")

=== ALL DATA SOURCES LOADED ===
WTI oil:                10436 rows
Natural gas:             7645 rows
Brent crude:             4584 rows
Gold:                    6357 rows
Copper:                  6362 rows
VIX:                     9066 rows
DXY dollar index:       10218 rows
GPR index:               1516 rows
ACLED violence:         28781 rows
ACLED fatalities:        2947 rows
ACLED MENA:            147304 rows
ACLED Europe/C.Asia:   120245 rows

Ready to start cleaning!


## 4. Master Date Spine and WTI Merge

The master dataset is built around a continuous daily date spine covering every calendar day from 2 January 1986 to 31 December 2025 — a total of 14,609 rows. This gives a fixed backbone that every other data source will be joined onto.

WTI crude oil is the first series merged in because it is the primary variable of interest and is available for the full date range. We use a left join so that every date in the spine is kept, then forward-fill any gaps (weekends and public holidays) with the most recent available price.

In [39]:
# MASTER DATE SPINE

date_spine = pd.DataFrame()
date_spine['observation_date'] = pd.date_range(start='1986-01-02', end='2025-12-31', freq='D')

print("Date spine created:", len(date_spine), "rows")

wti_clean = wti_raw.copy()
wti_clean.columns = ['observation_date', 'wti_price']
wti_clean['observation_date'] = pd.to_datetime(wti_clean['observation_date'])
wti_clean['wti_price'] = pd.to_numeric(wti_clean['wti_price'], errors='coerce')

master = pd.merge(date_spine, wti_clean, on='observation_date', how='left')
master['wti_price'] = master['wti_price'].ffill()

print("Master dataset shape after WTI:", master.shape)
print("WTI missing values:", master['wti_price'].isna().sum())
print(master[['observation_date', 'wti_price']].head(10))

Date spine created: 14609 rows
Master dataset shape after WTI: (14609, 2)
WTI missing values: 0
  observation_date  wti_price
0       1986-01-02     25.560
1       1986-01-03     26.000
2       1986-01-04     26.000
3       1986-01-05     26.000
4       1986-01-06     26.530
5       1986-01-07     25.850
6       1986-01-08     25.870
7       1986-01-09     26.030
8       1986-01-10     25.650
9       1986-01-11     25.650


## 5. Brent Crude Merge

Brent crude is downloaded from Yahoo Finance with timezone-aware timestamps, so we strip the timezone before merging. Prices are forward-filled to cover non-trading days. An availability flag is added so it is easy to filter the dataset to just the period where Brent data exists (from around 1990 onward).

In [40]:
brent_clean = brent_raw[['Close']].copy()
brent_clean = brent_clean.reset_index()
brent_clean.columns = ['observation_date', 'brent_price']
brent_clean['observation_date'] = pd.to_datetime(brent_clean['observation_date'])
brent_clean['observation_date'] = brent_clean['observation_date'].dt.tz_localize(None)

master = pd.merge(master, brent_clean, on='observation_date', how='left')
master['brent_price'] = master['brent_price'].ffill()
master['brent_data_available'] = master['brent_price'].notna().map({True: 'Yes', False: 'No'})

print("After Brent:", master.shape)
print("Brent available:", master['brent_data_available'].value_counts().to_dict())

After Brent: (14609, 4)
Brent available: {'No': 7879, 'Yes': 6730}


## 6. Gold, Copper, VIX, and DXY Merge

Gold, copper, the VIX, and the DXY Dollar Index are all cleaned and merged in the same block. The steps are the same for each: extract the closing price, reset the index so the date becomes a plain column, strip any timezone info, merge onto the master spine, and forward-fill weekend and holiday gaps. Availability flags are added for gold, copper, and VIX since those series start later than 1986.

In [41]:
gold_clean = gold_raw[['Close']].copy()
gold_clean = gold_clean.reset_index()
gold_clean.columns = ['observation_date', 'gold_price']
gold_clean['observation_date'] = pd.to_datetime(gold_clean['observation_date'])
gold_clean['observation_date'] = gold_clean['observation_date'].dt.tz_localize(None)

master = pd.merge(master, gold_clean, on='observation_date', how='left')
master['gold_price'] = master['gold_price'].ffill()
master['gold_data_available'] = master['gold_price'].notna().map({True: 'Yes', False: 'No'})

print("After Gold:", master.shape)
print("Gold available:", master['gold_data_available'].value_counts().to_dict())

########################################

copper_clean = copper_raw[['Close']].copy()
copper_clean = copper_clean.reset_index()
copper_clean.columns = ['observation_date', 'copper_price']
copper_clean['observation_date'] = pd.to_datetime(copper_clean['observation_date'])
copper_clean['observation_date'] = copper_clean['observation_date'].dt.tz_localize(None)

master = pd.merge(master, copper_clean, on='observation_date', how='left')
master['copper_price'] = master['copper_price'].ffill()
master['copper_data_available'] = master['copper_price'].notna().map({True: 'Yes', False: 'No'})

print("After Copper:", master.shape)
print("Copper available:", master['copper_data_available'].value_counts().to_dict())

########################################

vix_clean = vix_raw[['Close']].copy()
vix_clean = vix_clean.reset_index()
vix_clean.columns = ['observation_date', 'vix_close']
vix_clean['observation_date'] = pd.to_datetime(vix_clean['observation_date'])
vix_clean['observation_date'] = vix_clean['observation_date'].dt.tz_localize(None)

master = pd.merge(master, vix_clean, on='observation_date', how='left')
master['vix_close'] = master['vix_close'].ffill()
master['vix_data_available'] = master['vix_close'].notna().map({True: 'Yes', False: 'No'})

print("After VIX:", master.shape)
print("VIX available:", master['vix_data_available'].value_counts().to_dict())

########################################

dxy_clean = dxy_raw[['Close']].copy()
dxy_clean = dxy_clean.reset_index()
dxy_clean.columns = ['observation_date', 'dxy_index']
dxy_clean['observation_date'] = pd.to_datetime(dxy_clean['observation_date'])
dxy_clean['observation_date'] = dxy_clean['observation_date'].dt.tz_localize(None)

master = pd.merge(master, dxy_clean, on='observation_date', how='left')
master['dxy_index'] = master['dxy_index'].ffill()

print("After DXY:", master.shape)
print("DXY missing:", master['dxy_index'].isna().sum())

After Gold: (14609, 6)
Gold available: {'Yes': 9255, 'No': 5354}
After Copper: (14609, 8)
Copper available: {'Yes': 9255, 'No': 5354}
After VIX: (14609, 10)
VIX available: {'Yes': 13148, 'No': 1461}
After DXY: (14609, 11)
DXY missing: 0


## 7. Natural Gas Merge

Henry Hub natural gas prices are loaded from a FRED CSV file. The column is renamed, the date is parsed as a proper datetime, and any non-numeric values are forced to NaN. After merging onto the master spine, prices are forward-filled. Natural gas data only starts in 1997, so the first eleven years of the master dataset have NaN for this column, captured by the availability flag.

In [42]:
natgas_clean = natgas_raw.copy()
natgas_clean.columns = ['observation_date', 'natgas_price']
natgas_clean['observation_date'] = pd.to_datetime(natgas_clean['observation_date'])
natgas_clean['natgas_price'] = pd.to_numeric(natgas_clean['natgas_price'], errors='coerce')

master = pd.merge(master, natgas_clean, on='observation_date', how='left')
master['natgas_price'] = master['natgas_price'].ffill()
master['natgas_data_available'] = master['natgas_price'].notna().map({True: 'Yes', False: 'No'})

print("After Natural Gas:", master.shape)
print("Natgas available:", master['natgas_data_available'].value_counts().to_dict())
print("Natgas missing:", master['natgas_price'].isna().sum())

After Natural Gas: (14609, 13)
Natgas available: {'Yes': 10586, 'No': 4023}
Natgas missing: 4023


## 8. Geopolitical Risk (GPR) Index Merge

The GPR dataset is a monthly index measuring geopolitical risk based on how often conflict-related words appear in major newspaper articles. We keep the headline index (GPR), its two sub-components (threats vs. actual acts), and three country-level series for Russia, Saudi Arabia, and Israel — the three most geopolitically relevant actors for oil markets.

Because GPR is monthly but our master dataset is daily, we merge on year and month rather than on the exact date. Every day in a given month gets the same GPR value, which is the standard approach when using monthly indicators in a daily dataset.

In [43]:
gpr_clean = gpr_raw.copy()
gpr_clean = gpr_clean[['month', 'GPR', 'GPRT', 'GPRA', 'GPRC_RUS', 'GPRC_SAU', 'GPRC_ISR']].copy()
gpr_clean.columns = ['observation_date', 'gpr_index', 'gpr_threats', 'gpr_acts',
                     'gpr_russia', 'gpr_saudi', 'gpr_israel']
gpr_clean['observation_date'] = pd.to_datetime(gpr_clean['observation_date'])
gpr_clean = gpr_clean[gpr_clean['observation_date'] >= '1986-01-01']

print("GPR rows after filtering:", len(gpr_clean))
print("GPR date range:", gpr_clean['observation_date'].min(), "to", gpr_clean['observation_date'].max())
print("GPR missing values per column:")
print(gpr_clean.isna().sum())

GPR rows after filtering: 484
GPR date range: 1986-01-01 00:00:00 to 2026-04-01 00:00:00
GPR missing values per column:
observation_date    0
gpr_index           0
gpr_threats         0
gpr_acts            0
gpr_russia          0
gpr_saudi           0
gpr_israel          0
dtype: int64


In [44]:
gpr_clean['year'] = gpr_clean['observation_date'].dt.year
gpr_clean['month'] = gpr_clean['observation_date'].dt.month

gpr_merge = gpr_clean.drop(columns=['observation_date'])

master['year'] = master['observation_date'].dt.year
master['month'] = master['observation_date'].dt.month

master = pd.merge(master, gpr_merge, on=['year', 'month'], how='left')

print("After GPR:", master.shape)
print("GPR index missing:", master['gpr_index'].isna().sum())
print("GPR threats missing:", master['gpr_threats'].isna().sum())
print("GPR Russia missing:", master['gpr_russia'].isna().sum())

After GPR: (14609, 21)
GPR index missing: 0
GPR threats missing: 0
GPR Russia missing: 0


## 9. ACLED Conflict Data Merge

ACLED (Armed Conflict Location and Event Data Project) provides detailed records of political violence worldwide. Three derived series are merged onto the master dataset:

- **Global violence events**: Total political violence events recorded globally per month and the number of countries affected
- **Global fatalities**: Total reported conflict fatalities per year (the annual figure is applied to every day in that year)
- **MENA violence events**: Monthly violence events restricted to 21 Middle East and North Africa countries, which are most directly relevant to oil prices

The global and MENA series are built by aggregating the country-level events file, which gives better historical coverage (back to 1997) than the regional Excel files. An initial attempt using the MENA regional Excel file is also shown but was dropped because its coverage only starts from 2014.

In [45]:
acled_violence_clean = acled_violence_raw.copy()
acled_violence_clean['month_num'] = pd.to_datetime(acled_violence_clean['MONTH'], format='%B').dt.month
acled_violence_clean = acled_violence_clean.rename(columns={'YEAR': 'year'})

acled_global = acled_violence_clean.groupby(['year', 'month_num']).agg(
    total_violence_events = ('EVENTS', 'sum'),
    countries_affected = ('COUNTRY', 'nunique')
).reset_index()

acled_global = acled_global.rename(columns={'month_num': 'month'})

print("ACLED global shape after grouping:", acled_global.shape)
print("Date range:", acled_global['year'].min(), "to", acled_global['year'].max())
print(acled_global.head())

master = pd.merge(master, acled_global, on=['year', 'month'], how='left')

master['total_violence_events'] = master['total_violence_events'].fillna(0)
master['countries_affected'] = master['countries_affected'].fillna(0)

print("\nAfter ACLED global:", master.shape)
print("Violence events missing:", master['total_violence_events'].isna().sum())

ACLED global shape after grouping: (353, 4)
Date range: 1997 to 2026
   year  month  total_violence_events  countries_affected
0  1997      1                    184                  48
1  1997      2                    108                  48
2  1997      3                    182                  48
3  1997      4                    153                  48
4  1997      5                    175                  48

After ACLED global: (14609, 23)
Violence events missing: 0


In [46]:
acled_fatalities_clean = acled_fatalities_raw.copy()

acled_fatalities_clean = acled_fatalities_clean.rename(columns={
    'YEAR': 'year',
    'FATALITIES': 'total_fatalities'
})

acled_fatalities_global = acled_fatalities_clean.groupby('year').agg(
    total_fatalities = ('total_fatalities', 'sum')
).reset_index()

print("Fatalities shape after grouping:", acled_fatalities_global.shape)
print("Date range:", acled_fatalities_global['year'].min(), "to", acled_fatalities_global['year'].max())
print(acled_fatalities_global.head())

master = pd.merge(master, acled_fatalities_global, on='year', how='left')
master['total_fatalities'] = master['total_fatalities'].fillna(0)

print("\nAfter fatalities:", master.shape)
print("Fatalities missing:", master['total_fatalities'].isna().sum())

Fatalities shape after grouping: (30, 2)
Date range: 1997 to 2026
   year  total_fatalities
0  1997             26820
1  1998             70713
2  1999            161840
3  2000             23840
4  2001             26993

After fatalities: (14609, 24)
Fatalities missing: 0


In [47]:
acled_mena_clean = acled_mena_raw.copy()
acled_mena_clean['WEEK'] = pd.to_datetime(acled_mena_clean['WEEK'])
acled_mena_clean['year'] = acled_mena_clean['WEEK'].dt.year
acled_mena_clean['month'] = acled_mena_clean['WEEK'].dt.month

acled_mena_monthly = acled_mena_clean.groupby(['year', 'month']).agg(
    mena_violence_events = ('EVENTS', 'sum'),
    mena_fatalities = ('FATALITIES', 'sum')
).reset_index()

print("MENA monthly shape:", acled_mena_monthly.shape)
print("Date range:", acled_mena_monthly['year'].min(), "to", acled_mena_monthly['year'].max())
print(acled_mena_monthly.head())

master = pd.merge(master, acled_mena_monthly, on=['year', 'month'], how='left')

master['mena_violence_events'] = master['mena_violence_events'].fillna(0)
master['mena_fatalities'] = master['mena_fatalities'].fillna(0)

print("\nAfter MENA:", master.shape)
print("MENA missing:", master['mena_violence_events'].isna().sum())

MENA monthly shape: (137, 4)
Date range: 2014 to 2026
   year  month  mena_violence_events  mena_fatalities
0  2014     12                    15               27
1  2015      1                   339              327
2  2015      2                   258              359
3  2015      3                   472             1315
4  2015      4                   823             2658

After MENA: (14609, 26)
MENA missing: 0


In [48]:
mena_countries = [
    'Iraq', 'Iran', 'Saudi Arabia', 'Syria', 'Yemen', 'Kuwait',
    'Libya', 'Egypt', 'Jordan', 'Lebanon', 'Israel', 'Palestine',
    'Bahrain', 'Qatar', 'United Arab Emirates', 'Oman', 'Turkey',
    'Algeria', 'Tunisia', 'Morocco', 'Sudan'
]

acled_mena_filtered = acled_violence_raw[acled_violence_raw['COUNTRY'].isin(mena_countries)].copy()

acled_mena_filtered['month_num'] = pd.to_datetime(acled_mena_filtered['MONTH'], format='%B').dt.month

acled_mena_monthly = acled_mena_filtered.groupby(['YEAR', 'month_num']).agg(
    mena_violence_events = ('EVENTS', 'sum')
).reset_index()

acled_mena_monthly.columns = ['year', 'month', 'mena_violence_events']

print("MENA rebuilt shape:", acled_mena_monthly.shape)
print("Date range:", acled_mena_monthly['year'].min(), "to", acled_mena_monthly['year'].max())
print(acled_mena_monthly.head())

MENA rebuilt shape: (353, 3)
Date range: 1997 to 2026
   year  month  mena_violence_events
0  1997      1                    96
1  1997      2                    18
2  1997      3                    51
3  1997      4                    50
4  1997      5                    15


In [49]:
master = master.drop(columns=['mena_violence_events', 'mena_fatalities'])

master = pd.merge(master, acled_mena_monthly, on=['year', 'month'], how='left')

master['mena_violence_events'] = master['mena_violence_events'].fillna(0)

print("MENA fixed. Missing:", master['mena_violence_events'].isna().sum())


MENA fixed. Missing: 0


In [50]:
russia_fsu_countries = [
    'Russia', 'Ukraine', 'Belarus', 'Georgia', 'Armenia',
    'Azerbaijan', 'Kazakhstan', 'Uzbekistan', 'Turkmenistan',
    'Tajikistan', 'Kyrgyzstan', 'Moldova'
]

acled_russia_filtered = acled_violence_raw[acled_violence_raw['COUNTRY'].isin(russia_fsu_countries)].copy()

acled_russia_filtered['month_num'] = pd.to_datetime(acled_russia_filtered['MONTH'], format='%B').dt.month

acled_russia_monthly = acled_russia_filtered.groupby(['YEAR', 'month_num']).agg(
    russia_fsu_violence_events = ('EVENTS', 'sum')
).reset_index()

acled_russia_monthly.columns = ['year', 'month', 'russia_fsu_violence_events']

print("Russia/FSU shape:", acled_russia_monthly.shape)
print("Date range:", acled_russia_monthly['year'].min(), "to", acled_russia_monthly['year'].max())
print(acled_russia_monthly.head())

master = pd.merge(master, acled_russia_monthly, on=['year', 'month'], how='left')

master['russia_fsu_violence_events'] = master['russia_fsu_violence_events'].fillna(0)

print("\nAfter Russia/FSU:", master.shape)
print("Russia/FSU missing:", master['russia_fsu_violence_events'].isna().sum())

Russia/FSU shape: (101, 3)
Date range: 2018 to 2026
   year  month  russia_fsu_violence_events
0  2018      1                        1781
1  2018      2                        2068
2  2018      3                        1818
3  2018      4                        2052
4  2018      5                        2201

After Russia/FSU: (14609, 26)
Russia/FSU missing: 0


In [51]:
acled_russia_check = acled_violence_raw[acled_violence_raw['COUNTRY'].isin(russia_fsu_countries)].copy()

print("Countries found and their year ranges:")
for country in russia_fsu_countries:
    country_data = acled_russia_check[acled_russia_check['COUNTRY'] == country]
    if len(country_data) > 0:
        print(f"  {country}: {country_data['YEAR'].min()} to {country_data['YEAR'].max()}")
    else:
        print(f"  {country}: NOT FOUND in dataset")

Countries found and their year ranges:
  Russia: 2018 to 2026
  Ukraine: 2018 to 2026
  Belarus: 2018 to 2026
  Georgia: 2018 to 2026
  Armenia: 2018 to 2026
  Azerbaijan: 2018 to 2026
  Kazakhstan: 2018 to 2026
  Uzbekistan: 2018 to 2026
  Turkmenistan: 2018 to 2026
  Tajikistan: 2018 to 2026
  Kyrgyzstan: 2018 to 2026
  Moldova: 2018 to 2026


In [52]:
master = master.drop(columns=['russia_fsu_violence_events'])

print("Russia/FSU column dropped.")
print("Master shape now:", master.shape)
print("Columns:", master.columns.tolist())

Russia/FSU column dropped.
Master shape now: (14609, 25)
Columns: ['observation_date', 'wti_price', 'brent_price', 'brent_data_available', 'gold_price', 'gold_data_available', 'copper_price', 'copper_data_available', 'vix_close', 'vix_data_available', 'dxy_index', 'natgas_price', 'natgas_data_available', 'year', 'month', 'gpr_index', 'gpr_threats', 'gpr_acts', 'gpr_russia', 'gpr_saudi', 'gpr_israel', 'total_violence_events', 'countries_affected', 'total_fatalities', 'mena_violence_events']


## 10. Time Features

Several calendar-based features are added to help any model capture seasonal and cyclical patterns. These include the quarter, month name, season, and day of the week. A `price_era` label is also assigned, dividing the full dataset into eight named historical periods based on major events that shaped oil markets: the Gulf War, the post-9/11 boom, the financial crisis, the shale revolution, COVID, and the Ukraine war.

In [53]:
master['quarter'] = master['observation_date'].dt.quarter
master['month_name'] = master['observation_date'].dt.strftime('%B')

quarter_map = {1: 'Q1', 2: 'Q2', 3: 'Q3', 4: 'Q4'}
master['quarter_name'] = master['quarter'].map(quarter_map)

# Season
def get_season(month):
    if month in [12, 1, 2]:
        return 'Winter'
    elif month in [3, 4, 5]:
        return 'Spring'
    elif month in [6, 7, 8]:
        return 'Summer'
    else:
        return 'Autumn'

master['season'] = master['month'].apply(get_season)

# Day of week (0 = Monday, 6 = Sunday)
master['day_of_week'] = master['observation_date'].dt.dayofweek

# Day name (Monday, Tuesday etc)
master['day_name'] = master['observation_date'].dt.strftime('%A')

# Price era - historical periods that shaped oil markets
def get_price_era(date):
    if date < pd.Timestamp('1991-01-17'):
        return 'Pre-Gulf War'
    elif date < pd.Timestamp('2001-09-11'):
        return 'Post-Gulf War'
    elif date < pd.Timestamp('2008-09-15'):
        return 'Post-9/11 Boom'
    elif date < pd.Timestamp('2010-01-01'):
        return 'Financial Crisis'
    elif date < pd.Timestamp('2014-06-01'):
        return 'Shale Revolution'
    elif date < pd.Timestamp('2020-01-01'):
        return 'Low Price Era'
    elif date < pd.Timestamp('2022-02-24'):
        return 'COVID Era'
    else:
        return 'Ukraine War Era'

master['price_era'] = master['observation_date'].apply(get_price_era)

print("After time features:", master.shape)
print("Price era distribution:")
print(master['price_era'].value_counts())

After time features: (14609, 32)
Price era distribution:
price_era
Post-Gulf War       3890
Post-9/11 Boom      2561
Low Price Era       2040
Pre-Gulf War        1841
Shale Revolution    1612
Ukraine War Era     1407
COVID Era            785
Financial Crisis     473
Name: count, dtype: int64


## 11. Price and Volatility Features

A set of derived features are calculated from the WTI price series to capture momentum, recent trends, and volatility. These include the daily percentage return, one-day and seven-day lagged prices, a 30-day rolling average, a categorical price level label (Very Low through Very High), and a volatility category based on the size of each day’s move. The Brent-WTI spread and the gold-to-WTI ratio are also calculated here as cross-commodity signals.

In [54]:

# WTI daily return (percentage change day over day)
master['wti_daily_return'] = master['wti_price'].pct_change() * 100

# WTI 7 day lag (price 7 days ago)
master['wti_price_lag1'] = master['wti_price'].shift(1)
master['wti_price_lag7'] = master['wti_price'].shift(7)

# WTI 30 day rolling average
master['wti_rolling_mean_30'] = master['wti_price'].rolling(window=30).mean()

# WTI price level category based on price bins
def get_price_level(price):
    if price < 30:
        return 'Very Low'
    elif price < 60:
        return 'Low'
    elif price < 90:
        return 'Medium'
    elif price < 120:
        return 'High'
    else:
        return 'Very High'

master['wti_price_level'] = master['wti_price'].apply(get_price_level)

# WTI volatility category based on absolute daily return
def get_volatility_cat(ret):
    if pd.isna(ret):
        return 'Stable'
    abs_ret = abs(ret)
    if abs_ret < 1:
        return 'Stable'
    elif abs_ret < 2:
        return 'Low Volatility'
    elif abs_ret < 5:
        return 'High Volatility'
    else:
        return 'Extreme Volatility'

master['wti_volatility_cat'] = master['wti_daily_return'].apply(get_volatility_cat)

# Brent/WTI spread (only where brent data is available)
master['wti_brent_spread'] = master['wti_price'] - master['brent_price']

# Gold to WTI ratio (fear vs demand signal)
master['gold_wti_ratio'] = master['gold_price'] / master['wti_price']

print("After engineered features:", master.shape)
print("\nWTI price level distribution:")
print(master['wti_price_level'].value_counts())
print("\nWTI volatility distribution:")
print(master['wti_volatility_cat'].value_counts())
print("\nMissing values in new columns:")
print(master[['wti_daily_return', 'wti_price_lag1', 'wti_price_lag7',
              'wti_rolling_mean_30', 'wti_brent_spread', 'gold_wti_ratio']].isna().sum())

After engineered features: (14609, 40)

WTI price level distribution:
wti_price_level
Very Low     6066
Medium       3498
Low          3364
High         1581
Very High     100
Name: count, dtype: int64

WTI volatility distribution:
wti_volatility_cat
Stable                8719
Low Volatility        2757
High Volatility       2638
Extreme Volatility     495
Name: count, dtype: int64

Missing values in new columns:
wti_daily_return          1
wti_price_lag1            1
wti_price_lag7            7
wti_rolling_mean_30      29
wti_brent_spread       7879
gold_wti_ratio         5354
dtype: int64


## 12. Conflict Intensity and Commodity Ratio Features

A conflict intensity category is assigned based on the monthly global violence event count, grouping observations into Low, Medium, High, and Extreme bands. The copper-to-WTI ratio is calculated and then labelled using the corrected version of the function in the next cell, indicating whether industrial demand (copper) or energy supply risk (oil) is relatively stronger at any point in time.

In [55]:
# Backfill the first few rows of lags and rolling mean
# (only affects the very first rows where no prior data exists)
master['wti_daily_return'] = master['wti_daily_return'].fillna(0)
master['wti_price_lag1'] = master['wti_price_lag1'].bfill()
master['wti_price_lag7'] = master['wti_price_lag7'].bfill()
master['wti_rolling_mean_30'] = master['wti_rolling_mean_30'].bfill()

# ============================================================
# Based on total global violence events per month
# ============================================================

def get_conflict_intensity(events):
    if events == 0:
        return 'No Data'
    elif events < 500:
        return 'Low'
    elif events < 1500:
        return 'Medium'
    elif events < 3000:
        return 'High'
    else:
        return 'Extreme'

master['conflict_intensity'] = master['total_violence_events'].apply(get_conflict_intensity)

print("Conflict intensity distribution:")
print(master['conflict_intensity'].value_counts())

# Copper to WTI ratio (copper in USD/lb, WTI in USD/barrel)
master['copper_wti_ratio'] = master['copper_price'] / master['wti_price']

print("\nMaster shape:", master.shape)

Conflict intensity distribution:
conflict_intensity
Low        4748
No Data    4017
Extreme    3622
Medium     1823
High        399
Name: count, dtype: int64

Commodity relationship distribution:
commodity_relationship
Oil Dominant    9255
No Data         5354
Name: count, dtype: int64

Master shape: (14609, 43)


In [56]:

print("Copper/WTI ratio statistics:")
print(master['copper_wti_ratio'].describe())

print("\nSample rows where copper data is available:")
print(master[master['copper_data_available'] == 'Yes'][['observation_date', 'copper_price', 'wti_price', 'copper_wti_ratio']].head(10))

Copper/WTI ratio statistics:
count   9255.000
mean       0.044
std        0.015
min       -0.063
25%        0.033
50%        0.042
75%        0.052
max        0.254
Name: copper_wti_ratio, dtype: float64

Sample rows where copper data is available:
     observation_date  copper_price  wti_price  copper_wti_ratio
5354       2000-08-30         0.885     33.250             0.027
5355       2000-08-31         0.885     33.090             0.027
5356       2000-09-01         0.889     33.420             0.027
5357       2000-09-02         0.889     33.420             0.027
5358       2000-09-03         0.889     33.420             0.027
5359       2000-09-04         0.889     33.420             0.027
5360       2000-09-05         0.906     33.920             0.027
5361       2000-09-06         0.901     34.970             0.026
5362       2000-09-07         0.906     35.180             0.026
5363       2000-09-08         0.912     33.620             0.027


In [57]:
# The first version of this function used bins of 300, 600, and 900.
# That was wrong because the copper/WTI ratio is a unitless decimal (roughly 0.02 to 0.25),
# not a large number. Copper prices from yfinance are in USD per pound (~$2-5),
# and WTI is in USD per barrel (~$20-150), so the ratio is always well below 1.
# The corrected bins below match the actual observed range shown in the cell above.

def get_commodity_relationship(ratio):
    if pd.isna(ratio):
        return 'No Data'
    elif ratio < 0.03:
        return 'Oil Dominant'
    elif ratio < 0.05:
        return 'Balanced'
    elif ratio < 0.08:
        return 'Copper Strong'
    else:
        return 'Copper Dominant'

# Recalculate with corrected bins
master['commodity_relationship'] = master['copper_wti_ratio'].apply(get_commodity_relationship)

print("Commodity relationship distribution (corrected):")
print(master['commodity_relationship'].value_counts())

Commodity relationship distribution (corrected):
commodity_relationship
No Data            5354
Balanced           4842
Copper Strong      2644
Oil Dominant       1586
Copper Dominant     183
Name: count, dtype: int64


## 13. Final Column Selection

The master dataframe has accumulated extra intermediate columns from the data cleaning and rebuild steps above. Here we select only the 43 final columns we want to keep, organised into logical groups: time identifiers, WTI price features, other commodity prices, market sentiment indicators, geopolitical risk scores, and conflict statistics.

In [58]:
final_columns = [
    # Time
    'observation_date', 'year', 'month', 'month_name', 'quarter',
    'quarter_name', 'season', 'day_of_week', 'day_name',

    # WTI oil (primary target)
    'wti_price', 'wti_price_lag1', 'wti_price_lag7',
    'wti_rolling_mean_30', 'wti_daily_return', 'wti_price_level',
    'wti_volatility_cat', 'price_era',

    # Brent crude
    'brent_price', 'brent_data_available', 'wti_brent_spread',

    # Gold
    'gold_price', 'gold_data_available', 'gold_wti_ratio',

    # Copper
    'copper_price', 'copper_data_available',
    'copper_wti_ratio', 'commodity_relationship',

    # Natural gas
    'natgas_price', 'natgas_data_available',

    # Market sentiment
    'vix_close', 'vix_data_available', 'dxy_index',

    # Geopolitical risk
    'gpr_index', 'gpr_threats', 'gpr_acts',
    'gpr_russia', 'gpr_saudi', 'gpr_israel',

    # Conflict
    'total_violence_events', 'countries_affected',
    'total_fatalities', 'mena_violence_events',
    'conflict_intensity'
]

# Reorder the dataframe
master = master[final_columns]

print("Final master shape:", master.shape)
print("Final columns:", master.columns.tolist())


print("\nMissing values by column:")
missing = master.isna().sum()
missing = missing[missing > 0]
print(missing)

Final master shape: (14609, 43)
Final columns: ['observation_date', 'year', 'month', 'month_name', 'quarter', 'quarter_name', 'season', 'day_of_week', 'day_name', 'wti_price', 'wti_price_lag1', 'wti_price_lag7', 'wti_rolling_mean_30', 'wti_daily_return', 'wti_price_level', 'wti_volatility_cat', 'price_era', 'brent_price', 'brent_data_available', 'wti_brent_spread', 'gold_price', 'gold_data_available', 'gold_wti_ratio', 'copper_price', 'copper_data_available', 'copper_wti_ratio', 'commodity_relationship', 'natgas_price', 'natgas_data_available', 'vix_close', 'vix_data_available', 'dxy_index', 'gpr_index', 'gpr_threats', 'gpr_acts', 'gpr_russia', 'gpr_saudi', 'gpr_israel', 'total_violence_events', 'countries_affected', 'total_fatalities', 'mena_violence_events', 'conflict_intensity']

Missing values by column:
brent_price         7879
wti_brent_spread    7879
gold_price          5354
gold_wti_ratio      5354
copper_price        5354
copper_wti_ratio    5354
natgas_price        4023
vix_c

In [59]:
master.sample(20)

,observation_date,year,month,month_name,quarter,quarter_name,season,day_of_week,day_name,wti_price,wti_price_lag1,wti_price_lag7,wti_rolling_mean_30,wti_daily_return,wti_price_level,wti_volatility_cat,price_era,brent_price,brent_data_available,wti_brent_spread,gold_price,gold_data_available,gold_wti_ratio,copper_price,copper_data_available,copper_wti_ratio,commodity_relationship,natgas_price,natgas_data_available,vix_close,vix_data_available,dxy_index,gpr_index,gpr_threats,gpr_acts,gpr_russia,gpr_saudi,gpr_israel,total_violence_events,countries_affected,total_fatalities,mena_violence_events,conflict_intensity
10728,2015-05-18,2015,5,May,2,Q2,Spring,0,Monday,59.440,59.730,59.230,58.429,-0.486,Low,Stable,Low Price Era,66.270,Yes,-6.830,1227.800,Yes,20.656,2.930,Yes,0.049,Balanced,3.010,Yes,12.730,Yes,94.190,76.262,77.669,72.862,0.453,0.500,0.197,2210.000,61.000,62405.000,1161.000,High
5179,2000-03-08,2000,3,March,1,Q1,Spring,2,Wednesday,31.220,33.900,31.710,30.311,-7.906,Low,Extreme Volatility,Post-Gulf War,NaN,No,NaN,NaN,No,NaN,NaN,No,NaN,No Data,2.740,Yes,23.820,Yes,105.930,50.102,55.031,40.816,0.350,0.060,0.148,209.000,48.000,23840.000,60.000,Low
9896,2013-02-05,2013,2,February,1,Q1,Winter,1,Tuesday,96.680,96.210,97.620,95.342,0.489,High,Stable,Shale Revolution,116.520,Yes,-19.840,1672.400,Yes,17.298,3.761,Yes,0.039,Balanced,3.340,Yes,13.720,Yes,79.490,81.590,77.605,85.658,0.446,0.134,0.410,891.000,58.000,36481.000,112.000,Medium
764,1988-02-05,1988,2,February,1,Q1,Winter,4,Friday,17.340,17.170,16.970,17.033,0.990,Very Low,Stable,Pre-Gulf War,NaN,No,NaN,NaN,No,NaN,NaN,No,NaN,No Data,NaN,No,NaN,No,91.150,86.206,90.189,73.823,0.969,0.076,0.438,0.000,0.000,0.000,0.000,No Data
4608,1998-08-15,1998,8,August,3,Q3,Summer,5,Saturday,13.400,13.400,13.850,13.832,0.000,Very Low,Stable,Post-Gulf War,NaN,No,NaN,NaN,No,NaN,NaN,No,NaN,No Data,1.830,Yes,34.340,Yes,102.430,97.021,83.939,123.839,0.624,0.497,0.301,196.000,48.000,70713.000,16.000,Low
3966,1996-11-11,1996,11,November,4,Q4,Autumn,0,Monday,23.350,23.600,22.800,24.319,-1.059,Very Low,Low Volatility,Post-Gulf War,NaN,No,NaN,NaN,No,NaN,NaN,No,NaN,No Data,NaN,No,14.040,Yes,86.290,57.345,58.058,51.038,0.254,0.075,0.173,0.000,0.000,0.000,0.000,No Data
5569,2001-04-02,2001,4,April,2,Q2,Spring,0,Monday,25.700,26.370,27.400,27.132,-2.541,Very Low,High Volatility,Post-Gulf War,NaN,No,NaN,255.600,Yes,9.946,0.755,Yes,0.029,Oil Dominant,5.250,Yes,31.210,Yes,117.340,50.569,52.774,47.172,0.280,0.045,0.305,239.000,48.000,26993.000,26.000,Low
13234,2022-03-28,2022,3,March,1,Q1,Spring,0,Monday,107.550,116.200,112.140,108.757,-7.444,High,Extreme Volatility,Ukraine War Era,112.480,Yes,-4.930,1939.600,Yes,18.034,4.714,Yes,0.044,Balanced,5.520,Yes,19.630,Yes,99.090,318.955,403.714,250.956,8.801,0.457,0.597,10588.000,119.000,171020.000,2044.000,Extreme
4644,1998-09-20,1998,9,September,3,Q3,Autumn,6,Sunday,15.530,15.530,14.390,14.264,0.000,Very Low,Stable,Post-Gulf War,NaN,No,NaN,NaN,No,NaN,NaN,No,NaN,No Data,2.270,Yes,38.630,Yes,96.640,62.513,68.477,55.130,0.493,0.157,0.169,225.000,48.000,70713.000,32.000,Low
3333,1995-02-17,1995,2,February,1,Q1,Winter,4,Friday,18.940,18.630,18.470,18.451,1.664,Very Low,Low Volatility,Post-Gulf War,NaN,No,NaN,NaN,No,NaN,NaN,No,NaN,No Data,NaN,No,11.710,Yes,86.430,76.875,72.594,77.146,0.615,0.044,0.288,0.000,0.000,0.000,0.000,No Data


In [60]:
master.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 14609 entries, 0 to 14608
Data columns (total 43 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   observation_date        14609 non-null  datetime64[ns]
 1   year                    14609 non-null  int32         
 2   month                   14609 non-null  int32         
 3   month_name              14609 non-null  object        
 4   quarter                 14609 non-null  int32         
 5   quarter_name            14609 non-null  object        
 6   season                  14609 non-null  object        
 7   day_of_week             14609 non-null  int32         
 8   day_name                14609 non-null  object        
 9   wti_price               14609 non-null  float64       
 10  wti_price_lag1          14609 non-null  float64       
 11  wti_price_lag7          14609 non-null  float64       
 12  wti_rolling_mean_30     14609 non-null  float6

In [61]:
master.nunique()

observation_date          14609
year                         40
month                        12
month_name                   12
quarter                       4
quarter_name                  4
season                        4
day_of_week                   7
day_name                      7
wti_price                  5601
wti_price_lag1             5601
wti_price_lag7             5600
wti_rolling_mean_30       14147
wti_daily_return           9763
wti_price_level               5
wti_volatility_cat            4
price_era                     8
brent_price                3469
brent_data_available          2
wti_brent_spread           4428
gold_price                 5102
gold_data_available           2
gold_wti_ratio             6366
copper_price               4122
copper_data_available         2
copper_wti_ratio           6370
commodity_relationship        5
natgas_price                921
natgas_data_available         2
vix_close                  2522
vix_data_available            2
dxy_inde

## 14. Save the Dataset

The cleaned and merged master dataset is saved to a CSV file. A quick summary confirms the shape, date range, and split between trading days and weekend days. The row count check confirms we have well over 10,000 rows, which meets the data size requirement for this project.

In [62]:
# Save as CSV
master.to_csv('capstone_master_dataset_v2.csv', index=False)

print("Dataset saved successfully!")
print(f"Final shape: {master.shape[0]} rows x {master.shape[1]} columns")
print(f"Date range: {master['observation_date'].min().date()} to {master['observation_date'].max().date()}")
print(f"\nQuick summary:")
print(f"  Total days:        {master.shape[0]:>7}")
print(f"  Trading days only: {master[master['day_of_week'] < 5].shape[0]:>7}")
print(f"  Weekend days:      {master[master['day_of_week'] >= 5].shape[0]:>7}")
print(f"  Columns:           {master.shape[1]:>7}")
print(f"\nRow count requirement: {'PASSED' if master.shape[0] >= 10000 else 'FAILED'}")

Dataset saved successfully!
Final shape: 14609 rows x 43 columns
Date range: 1986-01-02 to 2025-12-31

Quick summary:
  Total days:          14609
  Trading days only:   10435
  Weekend days:         4174
  Columns:                43

Row count requirement: PASSED


In [64]:
print("=== DATASET SUMMARY ===\n")

print("TIME COVERAGE:")
print(f"  1986-1989 (Pre-Gulf War):    {master[master['price_era'] == 'Pre-Gulf War'].shape[0]} days")
print(f"  1991-2001 (Post-Gulf War):   {master[master['price_era'] == 'Post-Gulf War'].shape[0]} days")
print(f"  2001-2008 (Post-9/11 Boom):  {master[master['price_era'] == 'Post-9/11 Boom'].shape[0]} days")
print(f"  2008-2010 (Financial Crisis):{master[master['price_era'] == 'Financial Crisis'].shape[0]} days")
print(f"  2010-2014 (Shale Rev):       {master[master['price_era'] == 'Shale Revolution'].shape[0]} days")
print(f"  2014-2020 (Low Price Era):   {master[master['price_era'] == 'Low Price Era'].shape[0]} days")
print(f"  2020-2022 (COVID Era):       {master[master['price_era'] == 'COVID Era'].shape[0]} days")
print(f"  2022-2025 (Ukraine War Era): {master[master['price_era'] == 'Ukraine War Era'].shape[0]} days")

print("\nWTI PRICE STATS:")
print(f"  Min:  ${master['wti_price'].min():.2f}")
print(f"  Max:  ${master['wti_price'].max():.2f}")
print(f"  Mean: ${master['wti_price'].mean():.2f}")

print("\nGPR INDEX STATS:")
print(f"  Min:  {master['gpr_index'].min():.1f}")
print(f"  Max:  {master['gpr_index'].max():.1f}")
print(f"  Mean: {master['gpr_index'].mean():.1f}")

=== DATASET SUMMARY ===

TIME COVERAGE:
  1986-1989 (Pre-Gulf War):    1841 days
  1991-2001 (Post-Gulf War):   3890 days
  2001-2008 (Post-9/11 Boom):  2561 days
  2008-2010 (Financial Crisis):473 days
  2010-2014 (Shale Rev):       1612 days
  2014-2020 (Low Price Era):   2040 days
  2020-2022 (COVID Era):       785 days
  2022-2025 (Ukraine War Era): 1407 days

WTI PRICE STATS:
  Min:  $-36.98
  Max:  $145.31
  Mean: $48.12

GPR INDEX STATS:
  Min:  39.0
  Max:  512.5
  Mean: 102.7


## Summary

The saved file `capstone_master_dataset_v2.csv` contains **14,609 rows** and **43 columns**, covering every calendar day from **2 January 1986 to 31 December 2025**.

### What the dataset contains

| Category | Columns | Coverage |
|---|---|---|
| Time identifiers | Date, year, month, quarter, season, day of week, price era | Full (1986–2025) |
| WTI oil price | Price, 1-day lag, 7-day lag, 30-day rolling mean, daily return, price level, volatility category | Full (1986–2025) |
| Brent crude | Price, WTI-Brent spread, availability flag | Partial (from ~1990) |
| Gold | Price, gold/WTI ratio, availability flag | Partial (from ~2000) |
| Copper | Price, copper/WTI ratio, commodity relationship label, availability flag | Partial (from ~2000) |
| Natural gas | Henry Hub price, availability flag | Partial (from 1997) |
| Market sentiment | VIX volatility index, US Dollar Index (DXY) | VIX from ~1990, DXY full |
| Geopolitical risk | GPR headline index, threats sub-index, acts sub-index, Russia/Saudi/Israel scores | Full (1986–2025) |
| Conflict statistics | Global violence events, countries affected, annual fatalities, MENA violence events, conflict intensity | Partial (from 1997) |

### Key notes on missing values

Columns with partial coverage have NaN values for the years before that data source existed. The `_data_available` flag columns make it easy to filter down to any sub-period where a particular series is available. WTI price has no missing values across the full date range.